# **MAIN CODE**

In [ ]:
import sqlite3
import re
import pytz
import random
import pandas as pd
import ipywidgets as widgets
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from IPython.display import display, clear_output
from datetime import datetime, timedelta
from google.colab import drive
from calendar import monthrange
from dateutil.relativedelta import relativedelta

# Historical prototype used Google Drive for persistent storage.
# Disabled in this publication copy. Do not mount personal storage
# or run this notebook against real financial data.
# drive.mount('/content/drive')

# Database location intentionally unset in this publication copy.
# Configure a separate disposable test database only if deliberately
# reviewing the historical implementation. Do not use the desktop database.
db_path = None

# Global variable to store the current logged-in user
current_user = None

# --- Database Setup --- (Add streak columns and new tables)
def create_db():
    conn = sqlite3.connect(db_path)  # Use the path on Google Drive
    cursor = conn.cursor()

    # Add columns if they don't exist
    try:
        cursor.execute('ALTER TABLE users ADD COLUMN last_login_date TEXT;')
        conn.commit()  # Commit after the first statement
        cursor.execute('ALTER TABLE users ADD COLUMN streak_count INTEGER DEFAULT 0;')
        conn.commit()  # Commit after the second statement
    except sqlite3.OperationalError:
        # The columns already exist, continue as usual
        pass

    # Create tables for users, accounts, budgets, transactions, subscriptions, and wishlist
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS users (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        username TEXT UNIQUE NOT NULL,
        email TEXT UNIQUE NOT NULL,
        password TEXT NOT NULL,
        last_login_date TEXT,
        streak_count INTEGER DEFAULT 0
    )
    ''')

    cursor.execute('''
    CREATE TABLE IF NOT EXISTS accounts (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        user_id INTEGER,
        account_name TEXT NOT NULL,
        balance REAL NOT NULL,
        notes TEXT,
        FOREIGN KEY(user_id) REFERENCES users(id)
    )
    ''')

    cursor.execute('''
    CREATE TABLE IF NOT EXISTS budgets (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        user_id INTEGER,
        budget_name TEXT NOT NULL,
        timeframe TEXT NOT NULL,
        start_date TEXT NOT NULL,
        end_date TEXT NOT NULL,
        category TEXT,
        amount REAL NOT NULL,
        spent REAL DEFAULT 0,
        FOREIGN KEY(user_id) REFERENCES users(id)
    )
    ''')

    cursor.execute('''
    CREATE TABLE IF NOT EXISTS transactions (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        user_id INTEGER,
        account_id INTEGER,
        type TEXT NOT NULL,  -- 'income' or 'expense'
        category TEXT NOT NULL,
        amount REAL NOT NULL,
        date TEXT NOT NULL,
        time TEXT NOT NULL,
        FOREIGN KEY(user_id) REFERENCES users(id),
        FOREIGN KEY(account_id) REFERENCES accounts(id)
    )
    ''')

    cursor.execute('''
    CREATE TABLE IF NOT EXISTS categories (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        user_id INTEGER,
        type TEXT NOT NULL,  -- 'income' or 'expense'
        name TEXT NOT NULL,
        UNIQUE(user_id, type, name),
        FOREIGN KEY(user_id) REFERENCES users(id)
    )
    ''')

    cursor.execute('''
    CREATE TABLE IF NOT EXISTS subscriptions (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        user_id INTEGER,
        subscription_name TEXT NOT NULL,
        category TEXT NOT NULL,
        amount REAL NOT NULL,
        charge_dates TEXT NOT NULL,  -- Store dates as comma-separated
        FOREIGN KEY(user_id) REFERENCES users(id)
    )
    ''')

    cursor.execute('''
    CREATE TABLE IF NOT EXISTS wishlist (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        user_id INTEGER,
        item_name TEXT NOT NULL,
        price REAL NOT NULL,
        notes TEXT,
        link TEXT,
        progress REAL DEFAULT 0,
        FOREIGN KEY(user_id) REFERENCES users(id)
    )
    ''')

    cursor.execute('''
        CREATE TABLE IF NOT EXISTS badges (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            user_id INTEGER,
            badge_name TEXT NOT NULL,
            badge_description TEXT,
            date_awarded TEXT NOT NULL,
            FOREIGN KEY(user_id) REFERENCES users(id)
        )
    ''')

    conn.commit()
    conn.close()

# --- Validation Helpers ---
def is_valid_username(username):
    return 6 <= len(username) <= 10

def is_valid_email(email):
    return email.endswith("@gmail.com")

def is_valid_password(password):
    return (
        len(password) >= 8 and
        re.search(r'[A-Z]', password) and
        re.search(r'\d', password) and
        re.search(r'[!@#$%^&*(),.?":{}|<>_\-+=~`\\/\[\];]', password)
    )

def username_exists(username):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM users WHERE username = ?", (username,))
    exists = cursor.fetchone() is not None
    conn.close()
    return exists

# --- Auth Functions ---
def register_user(username, email, password):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    try:
        cursor.execute('''
        INSERT INTO users (username, email, password, last_login_date, streak_count)
        VALUES (?, ?, ?, ?, ?)
        ''', (username, email, password, datetime.today().strftime("%Y-%m-%d"), 0))
        conn.commit()
        return True, "✅ Registration successful! Go back and login."
    except sqlite3.IntegrityError:
        return False, "❌ Username or email already exists."
    finally:
        conn.close()

# --- Auth Functions ---
def login_user(username, password):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute('''
    SELECT * FROM users WHERE username = ? AND password = ?
    ''', (username, password))
    user = cursor.fetchone()

    if user:
        # Update streak after successful login
        update_streak(user[0])

        # Award badges after login (not during registration)
        check_and_award_badges(user[0])  # Check and award badges after login

    conn.close()
    return user

# --- Streak Logic ---
def update_streak(user_id):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Get the current date and last login date
    current_date = datetime.today().strftime("%Y-%m-%d")
    cursor.execute("SELECT last_login_date, streak_count FROM users WHERE id = ?", (user_id,))
    last_login_date, streak_count = cursor.fetchone()

    # Handle the case where last_login_date is None (for new users)
    if last_login_date is None:
        last_login_date = current_date
        streak_count = 1  # Start the streak count

    if last_login_date != current_date:
        # If last login wasn't today, check if it was yesterday
        last_login_datetime = datetime.strptime(last_login_date, "%Y-%m-%d")
        yesterday = (datetime.today() - timedelta(days=1)).strftime("%Y-%m-%d")

        if last_login_date == yesterday:
            # Increment streak if last login was yesterday
            streak_count += 1
        else:
            # Reset streak if the user missed a day
            streak_count = 1

        # Update streak and last login date
        cursor.execute("UPDATE users SET last_login_date = ?, streak_count = ? WHERE id = ?",
                       (current_date, streak_count, user_id))
        conn.commit()

    conn.close()

    # Return the streak count
    return streak_count

# List of financial theory quotes
quotes = [
    "Save 3-6 months' worth of living expenses to provide a safety net in case of emergencies. (Emergency Fund Rule)",
    "Allocate your income with 50% for needs, 30% for wants, and 20% for savings and debt repayment. (The 50/30/20 Budgeting Rule)",
    "Prioritize saving a portion of your income before spending on anything else. (Pay Yourself First Principle)",
    "Pay off your smallest debts first to gain momentum and motivation before tackling larger debts. (The Debt Snowball Method)",
    "Estimate how long it will take for an investment to double by dividing 72 by the annual rate of return. (Rule of 72)",
    "Money today is worth more than the same amount in the future due to its potential to earn interest. (The Time Value of Money)",
    "Subtract your age from 100 to determine the percentage of your portfolio that should be in stocks. (The Rule of 100 for Retirement Savings)",
    "Withdraw 4% annually from your retirement savings to ensure it lasts throughout retirement. (The 4% Rule for Retirement Withdrawal)",
    "Small, recurring expenses like daily coffee can add up over time, affecting your long-term financial health. (The Latte Factor)",
    "Diversify your investments across different asset classes to reduce risk and enhance returns. (Asset Allocation)",
    "Aim to save 10% of your annual income for long-term goals like retirement. (The Rule of 10 for Savings Goal)",
    "Focus on the 20% of your spending that makes up 80% of your expenses to maximize financial savings. (The 80/20 Rule or Pareto Principle)",
    "Acquiring new things often leads to wanting more, which can cause unnecessary spending. (The Diderot Effect)",
    "Avoid continuing to invest in a losing project due to the money already spent and instead cut your losses. (The Sunk Cost Fallacy)",
    "Estimate how long it will take for your investment to double by dividing 115 by the annual interest rate. (The Rule of 115 for Doubling Money with Interest)",
    "Divide your retirement savings into short-term, medium-term, and long-term buckets based on risk and liquidity. (The 3-Bucket Strategy)",
    "Calculate the ratio of your monthly debt payments to your monthly income to assess how much debt you can afford. (The Debt-to-Income Ratio or DTI)",
    "Invest a fixed amount regularly, regardless of market conditions, to reduce the impact of market volatility. (Dollar-Cost Averaging)",
    "Dedicate 5 hours a week to learning and improving your financial literacy to make better financial decisions. (The 5-Hour Rule)",
    "Recognize cognitive biases like overconfidence and loss aversion that can lead to poor financial decisions and manage them effectively. (The Behavioral Biases Theory)"
]

# Function to show the welcome message and the random quote
def show_welcome_message():
    # Choose a random quote
    selected_quote = random.choice(quotes)

    # Create widgets for the welcome message and the quote
    quote_widget = widgets.HTML(value=f"<p><i>{selected_quote}</i></p>")

    # Create a container to display them in order: Welcome Message -> Quote -> Options
    container = widgets.VBox([quote_widget])

    # Display the container
    display(container)

# --- UI Components --- (Main Menu)
def show_main_menu(user):
    global current_user
    current_user = user  # Set current user when entering the main menu

    clear_output()

    user_id, username, email = user[0], user[1], user[2]
    today = datetime.today().strftime("%A, %d %B %Y")

    # Update streak
    streak_count = update_streak(user_id)

    # Motivational sentence
    if streak_count == 0:
        motivation = "🚀 Welcome to the journey! Build a streak that you can be proud of!"
    elif streak_count == 1:
        motivation = "🔥 You've started your streak! Keep it up!"
    elif streak_count <= 5:
        motivation = "🔥 You're on fire! Let's keep this streak going!"
    elif streak_count <= 10:
        motivation = "🔥 Amazing progress! You're unstoppable!"
    else:
        motivation = "🔥 You're a streak master! Keep pushing yourself!"

    # Welcome message
    tk_title = widgets.HTML(value=f"<h3>Welcome, {username}!</h3><p>{email}</p><p>Today is: <b>{today}</b></p>")

    # Display streak count and motivation
    streak_label = widgets.HTML(value=f"<h4>Streak: {streak_count} days {motivation}</h4>")

    # Features list
    features = [
        "1. Account Balance",
        "2. Budgets",
        "3. Add Transaction",
        "4. Add Subscription",
        "5. Wishlist",
        "6. Dashboard",
        "7. Badges"
    ]

    # Create buttons for features
    feature_buttons = [
        widgets.Button(description=f, layout=widgets.Layout(width="auto"))
        for f in features
    ]

    # Assign event handlers for feature buttons
    feature_buttons[0].on_click(lambda b: show_account_balance(user_id))
    feature_buttons[1].on_click(lambda b: show_budgets(user_id))
    feature_buttons[2].on_click(lambda b: show_add_transaction(user_id))
    feature_buttons[3].on_click(lambda b: show_add_subscription(user_id))
    feature_buttons[4].on_click(lambda b: show_wishlist(user_id))
    feature_buttons[5].on_click(lambda b: show_dashboard(user_id))
    feature_buttons[6].on_click(lambda b: show_badges(user_id))

    # Create a VBox container to display all components aligned left
    main_container = widgets.VBox([tk_title, streak_label] + feature_buttons, layout=widgets.Layout(align_items='flex-start'))

    # Display everything together in a single container
    display(main_container)

# --- Authentication Choice Screen ---
def show_auth_choice_screen():
    clear_output()

    login_btn = widgets.Button(description="Login", button_style='success')
    register_btn = widgets.Button(description="Register", button_style='info')

    def on_login_click(b):
        show_login_screen()

    def on_register_click(b):
        show_register_screen()

    login_btn.on_click(on_login_click)
    register_btn.on_click(on_register_click)

    display(widgets.HTML(value="<h2>Welcome to the Budgeting App</h2><p>Please choose an option:</p>"),
            login_btn, register_btn)

    show_welcome_message()

# --- Login Screen ---
def show_login_screen():
    clear_output()

    username_input = widgets.Text(placeholder="Enter username", description="Username:")
    password_input = widgets.Password(placeholder="Enter password", description="Password:")

    login_button = widgets.Button(description="Login", button_style='success')
    back_button = widgets.Button(description="Back")

    output = widgets.Output()

    def on_login(b):
        user = login_user(username_input.value, password_input.value)
        with output:
            clear_output()
            if user:
                show_main_menu(user)
            else:
                print("❌ Invalid username or password.")

    login_button.on_click(on_login)
    back_button.on_click(lambda b: show_auth_choice_screen())

    display(widgets.HTML(value="<h3>Login</h3>"),
            username_input, password_input,
            login_button, back_button, output)

# --- Register Screen ---
def show_register_screen():
    clear_output()

    username_input = widgets.Text(placeholder="Choose username", description="Username:")
    email_input = widgets.Text(placeholder="Enter email", description="Email:")
    password_input = widgets.Password(placeholder="Create password", description="Password:")

    register_button = widgets.Button(description="Register", button_style='info')
    back_button = widgets.Button(description="Back")

    output = widgets.Output()

    def on_register(b):
        with output:
            clear_output()

            username = username_input.value.strip()
            email = email_input.value.strip()
            password = password_input.value

            if not is_valid_username(username):
                print("❌ Username must be between 6 to 10 characters.")
                return

            if username_exists(username):
                print("❌ Username already exists. Please choose another.")
                return

            if not is_valid_email(email):
                print("❌ Email must be a valid Gmail address (e.g., user@gmail.com).")
                return

            if not is_valid_password(password):
                print("❌ Password must be at least 8 characters and include:")
                print("- At least 1 capital letter")
                print("- At least 1 number")
                print("- At least 1 symbol (e.g., @, *, !, etc.)")
                print("✅ Example: Secure*123")
                return

            success, msg = register_user(username, email, password)
            print(msg)

    register_button.on_click(on_register)
    back_button.on_click(lambda b: show_auth_choice_screen())

    display(widgets.HTML(value="<h3>Register</h3>"),
            username_input, email_input, password_input,
            register_button, back_button, output)

# Show account balance
def show_account_balance(user_id):
    clear_output()

    accounts = view_accounts(user_id)

    if not accounts:
        display(widgets.HTML(value="<b>No accounts found. Please add an account.</b>"))

    # Check if user has reached the limit of 5 accounts
    if len(accounts) >= 5:
        display(widgets.HTML(value="<b>You have reached the maximum of 5 accounts! You cannot add more accounts.</b>"))

    # Display existing accounts
    for account in accounts:
        display(widgets.HTML(value=f"<b>{account[1]}</b> | Balance: {account[2]} | Notes: {account[3]}"))

    # One "Delete" button to delete an account
    delete_button = widgets.Button(description="Delete Account")
    delete_button.on_click(lambda b: show_delete_confirmation(user_id))

    # If user has fewer than 5 accounts, show the "Add Account" button
    if len(accounts) < 5:
        add_account_button = widgets.Button(description="Add Account")
        add_account_button.on_click(lambda b: show_add_account(user_id))

        # Display the buttons in the desired order: Add Account button first, then Delete button
        display(add_account_button, delete_button)
    else:
        display(delete_button)

    # Back to main menu button
    back_button = widgets.Button(description="Back to Main Menu")
    back_button.on_click(lambda b: show_main_menu(current_user))  # Use current_user here
    display(back_button)

# View Accounts Function
def view_accounts(user_id):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute('''SELECT id, account_name, balance, notes FROM accounts WHERE user_id = ?''', (user_id,))
    accounts = cursor.fetchall()
    conn.close()

    # Return accounts as a list of tuples (account_id, account_name, balance, notes)
    return accounts

# Function to show delete confirmation
def show_delete_confirmation(user_id):
    clear_output()

    accounts = view_accounts(user_id)

    # Show a dropdown of account names for the user to choose
    account_names = [account[1] for account in accounts]
    delete_account_dropdown = widgets.Dropdown(
        options=account_names,
        description="Choose Account:",
        disabled=False
    )

    # Confirmation buttons
    confirm_button = widgets.Button(description="Confirm Delete")
    cancel_button = widgets.Button(description="Cancel")

    # Handle delete account logic
    def on_confirm_delete(b):
        selected_account_name = delete_account_dropdown.value
        account_id = next(account[0] for account in accounts if account[1] == selected_account_name)
        delete_account(account_id, user_id)
        show_account_balance(user_id)  # Refresh account balance after deletion

    def on_cancel_delete(b):
        show_account_balance(user_id)  # Go back to account balance if canceled

    confirm_button.on_click(on_confirm_delete)
    cancel_button.on_click(on_cancel_delete)

    display(delete_account_dropdown, confirm_button, cancel_button)

# Delete account function
def delete_account(account_id, user_id):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute('DELETE FROM accounts WHERE id = ?', (account_id,))
    conn.commit()
    conn.close()

# Show add account form
def show_add_account(user_id):
    clear_output()

    account_name_input = widgets.Text(description="Account Name:")
    balance_input = widgets.FloatText(description="Balance:")
    notes_input = widgets.Text(description="Notes:")
    submit_button = widgets.Button(description="Add Account")

    # Check for duplicate account names before allowing addition
    def is_unique_account_name(account_name):
        accounts = view_accounts(user_id)
        return account_name not in [account[2] for account in accounts]

    def on_submit(b):
        account_name = account_name_input.value
        if not is_unique_account_name(account_name):
            display(widgets.HTML(value="<b>Account name already exists. Please choose a different name.</b>"))
        else:
            add_account(user_id, account_name, balance_input.value, notes_input.value)

    submit_button.on_click(on_submit)
    display(account_name_input, balance_input, notes_input, submit_button)

    back_button = widgets.Button(description="Back to Account Balance")
    back_button.on_click(lambda b: show_account_balance(user_id))  # Go back to account balance
    display(back_button)

# Add account function
def add_account(user_id, account_name, balance, notes):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute('''
    INSERT INTO accounts (user_id, account_name, balance, notes)
    VALUES (?, ?, ?, ?)
    ''', (user_id, account_name, balance, notes))
    conn.commit()
    conn.close()
    show_account_balance(user_id)  # Refresh the account balance screen

#Show Budgets
def show_budgets(user_id):
    clear_output()

    # Fetch all budgets for the user
    budgets = view_budgets(user_id)

    # Display message if no budgets are found
    if not budgets:
        display(widgets.HTML(value="<b>No budgets found. Please add a budget.</b>"))

    # Display existing budgets with progress bars and reminders
    if budgets:
        for budget in budgets:
            progress = calculate_budget_progress(budget[0], user_id)
            progress_bar = widgets.FloatProgress(value=progress, min=0, max=100, description="Progress:")

            # Display budget details
            display(widgets.HTML(value=f"<b>{budget[2]}</b> | Timeframe: {budget[3]} | Amount: {budget[7]}"))
            display(progress_bar)

            # Reminders for nearing or exceeding budget
            if progress >= 100:
                display(widgets.HTML(value="<b>Alert:</b> Budget has been exceeded. Please save money!"))
            elif progress >= 90:
                display(widgets.HTML(value="<b>Warning:</b> Budget is almost reached. Be careful with your spending!"))
            else:
                display(widgets.HTML(value="<b>All good!</b> Keep it up!"))

    # Show the "Add Budget" button if the user hasn't reached the maximum of 5 budgets
    if len(budgets) < 5:
        add_budget_button = widgets.Button(description="Add Budget")
        add_budget_button.on_click(lambda b: show_add_budget(user_id))
        display(add_budget_button)  # Display Add Budget button after existing budgets

    # Show the "Delete Budget" button if there are existing budgets
    if budgets:
        delete_budget_button = widgets.Button(description="Delete Budget")
        delete_budget_button.on_click(lambda b: show_delete_budget_dropdown(user_id, budgets))
        display(delete_budget_button)  # Display Delete Budget button after Add Budget

    # Display the "Back to Main Menu" button at the end
    back_button = widgets.Button(description="Back to Main Menu")
    back_button.on_click(lambda b: show_main_menu(current_user))  # Use current_user here
    display(back_button)

# View Budgets Function
def view_budgets(user_id):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute('''
    SELECT * FROM budgets WHERE user_id = ?
    ''', (user_id,))
    budgets = cursor.fetchall()
    conn.close()
    return budgets

# Function to show delete budget dropdown
def show_delete_budget_dropdown(user_id, budgets):
    clear_output()

    # Extract budget names
    budget_names = [budget[2] for budget in budgets]

    delete_budget_dropdown = widgets.Dropdown(
        options=budget_names,
        description="Choose Budget:",
        disabled=False
    )

    # Confirmation buttons
    confirm_button = widgets.Button(description="Confirm Delete")
    cancel_button = widgets.Button(description="Cancel")

    # Handle delete budget logic
    def on_confirm_delete(b):
        selected_budget_name = delete_budget_dropdown.value
        budget_id = next(budget[0] for budget in budgets if budget[2] == selected_budget_name)
        delete_budget(budget_id, user_id)
        show_budgets(user_id)  # Refresh the budget list after deletion

    def on_cancel_delete(b):
        show_budgets(user_id)  # Go back to the budget view if canceled

    confirm_button.on_click(on_confirm_delete)
    cancel_button.on_click(on_cancel_delete)

    display(delete_budget_dropdown, confirm_button, cancel_button)

# Delete budget function
def delete_budget(budget_id, user_id):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute('DELETE FROM budgets WHERE id = ?', (budget_id,))
    conn.commit()
    conn.close()

def calculate_budget_progress(budget_id, user_id):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Fetch the budget details (category, amount, timeframe, start date)
    cursor.execute("SELECT category, amount, timeframe, start_date FROM budgets WHERE id = ?", (budget_id,))
    row = cursor.fetchone()
    if not row:
        return 0
    category, total_budget, timeframe, start_date = row
    start_date = datetime.strptime(start_date, "%Y-%m-%d")
    today = datetime.today()

    # Determine the timeframe window
    if timeframe == "Daily":
        window_start = today.replace(hour=0, minute=0, second=0, microsecond=0)
    elif timeframe == "Weekly":
        window_start = today - timedelta(days=today.weekday())  # Start of this week
    elif timeframe == "Monthly":
        window_start = today.replace(day=1)  # First day of this month
    elif timeframe == "Yearly":
        window_start = today.replace(month=1, day=1)  # First day of this year
    else:
        window_start = start_date  # For any other cases

    window_start_str = window_start.strftime("%Y-%m-%d")

    # If the category is "General", sum up all expense transactions
    if category.lower() == 'general':
        cursor.execute('''
            SELECT SUM(amount) FROM transactions
            WHERE user_id = ? AND type = 'expense' AND date >= ?
        ''', (user_id, window_start_str))
    else:
        cursor.execute('''
            SELECT SUM(amount) FROM transactions
            WHERE user_id = ? AND type = 'expense' AND category = ? AND date >= ?
        ''', (user_id, category, window_start_str))

    total_spent = cursor.fetchone()[0]
    conn.close()

    if total_spent is None:
        total_spent = 0

    # Calculate progress as percentage
    progress = (total_spent / total_budget) * 100 if total_budget else 0

    # Ensure progress is capped at 100%
    return min(progress, 100)

def get_expense_categories(user_id):
    """Fetch distinct expense categories for the user, including 'General'."""
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Fetch all distinct expense categories
    cursor.execute("SELECT DISTINCT name FROM categories WHERE user_id = ? AND type = 'expense'", (user_id,))
    categories = [row[0] for row in cursor.fetchall()]

    # Always include 'General' in the list of expense categories
    if 'General' not in categories:
        categories.append('General')

    conn.close()
    return categories  # Return the updated categories list

def show_add_budget(user_id):
    clear_output()

    budget_name_input = widgets.Text(description="Budget Name:")
    timeframe_input = widgets.Dropdown(description="Timeframe:", options=['Daily', 'Weekly', 'Monthly', 'Yearly'])
    amount_input = widgets.FloatText(description="Amount:")
    category_input = widgets.Dropdown(
    description="Category:",
    options=get_expense_categories(user_id)  # Ensure 'General' is always included
    )
    output = widgets.Output()
    submit_button = widgets.Button(description="Submit Budget")

    def on_submit(b):
        output.clear_output()
        with output:
            if amount_input.value <= 0:
                print("❌ Budget amount must be greater than 0. Example: 1000.0")
                return

        # Clear form screen before updating
        clear_output()

        # Add budget and go back to budget list
        add_budget(
            user_id,
            budget_name_input.value,
            timeframe_input.value,
            category_input.value,
            amount_input.value
        )

        # Then refresh the budgets list
        show_budgets(user_id)

    submit_button.on_click(on_submit)

    display(widgets.VBox([
        budget_name_input,
        timeframe_input,
        amount_input,
        category_input,
        submit_button,
        output
    ]))

    back_button = widgets.Button(description="Back to Main Menu")
    back_button.on_click(lambda b: show_main_menu(current_user))
    display(back_button)

def add_budget(user_id, budget_name, timeframe, category, amount):
    start_date = datetime.today().date()

    # Automatically calculate the end date
    if timeframe == 'Daily':
        end_date = start_date + timedelta(days=1)
    elif timeframe == 'Weekly':
        end_date = start_date + timedelta(weeks=1)
    elif timeframe == 'Monthly':
        end_date = start_date + relativedelta(months=1)
    elif timeframe == 'Yearly':
        end_date = start_date + relativedelta(years=1)
    else:
        end_date = start_date

    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute('''
        INSERT INTO budgets (user_id, budget_name, timeframe, start_date, end_date, category, amount)
        VALUES (?, ?, ?, ?, ?, ?, ?)
    ''', (user_id, budget_name, timeframe, start_date.strftime("%Y-%m-%d"), end_date.strftime("%Y-%m-%d"), category, amount))
    conn.commit()
    conn.close()

    show_budgets(user_id)  # Refresh budgets

# Show add transaction
def show_add_transaction(user_id):
    clear_output(wait=True)  # Clear previous output, but avoid multiple clearing

    # Display the main header only once
    display(widgets.HTML(value="<h3>Add Transaction</h3>"))

    transaction_type = widgets.ToggleButtons(
        options=['Income', 'Expense'],
        description='Type:',
        disabled=False
    )

    category_dropdown = widgets.Dropdown(description="Category:")
    new_category_input = widgets.Text(placeholder="Enter new category name")

    amount_input = widgets.FloatText(description="Amount:")

    accounts = view_accounts(user_id)
    account_dropdown = widgets.Dropdown(
        options=[(f"{acc[1]} | Balance: {acc[2]}", acc[0]) for acc in accounts],
        description="Account:"
    )

    add_category_button = widgets.Button(description="Add Category", button_style='info')  # Only colored button
    submit_button = widgets.Button(description="Submit Transaction")  # Default style
    back_button = widgets.Button(description="Back to Main Menu")  # Default style

    output = widgets.Output()

    # New buttons for Category List and Transaction History
    show_category_button = widgets.Button(description="Category List", button_style='info')
    show_history_button = widgets.Button(description="Transaction History", button_style='info')

    # Event handlers for buttons
    def on_show_category_click(b):
        show_category_list(user_id)

    def on_show_history_click(b):
        show_transaction_history(user_id)

    def update_category_options():
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        cursor.execute(
            "SELECT name FROM categories WHERE user_id = ? AND type = ?",
            (user_id, transaction_type.value.lower())
        )
        categories = [row[0] for row in cursor.fetchall()]
        conn.close()
        category_dropdown.options = categories + ['+ Add Category'] if categories else ['+ Add Category']

    def on_type_change(change):
        if change['name'] == 'value':
            update_category_options()

    def on_add_category_clicked(b):
        new_cat = new_category_input.value.strip()
        if new_cat:
            conn = sqlite3.connect(db_path)
            cursor = conn.cursor()
            cursor.execute("SELECT 1 FROM categories WHERE user_id = ? AND type = ? AND name = ?",
                           (user_id, transaction_type.value.lower(), new_cat))
            exists = cursor.fetchone()
            if exists:
                print("❌ Category already exists.")
            else:
                cursor.execute('''
                    INSERT INTO categories (user_id, type, name)
                    VALUES (?, ?, ?)
                ''', (user_id, transaction_type.value.lower(), new_cat))
                conn.commit()
                print("✅ Category added.")
            conn.close()
            update_category_options()
            category_dropdown.value = new_cat
            new_category_input.value = ""

    def on_submit_clicked(b):
        with output:
            clear_output(wait=True)  # Clear only output, not the entire UI
            if amount_input.value <= 0:
                print("❌ Amount must be greater than 0.")
                return
            category = category_dropdown.value
            if category == '+ Add Category':
                print("❌ Please add and select a category before submitting.")
                return
            add_transaction(
                user_id=user_id,
                category=category,
                type=transaction_type.value.lower(),
                amount=amount_input.value,
                account_id=account_dropdown.value
            )
            print("✅ Transaction added successfully.")

            # Instead of re-calling the entire form, show a success message
            display(widgets.HTML(value="<h4>Transaction successfully added!</h4>"))
            display(back_button)  # Allow user to return to the main menu

    def on_back_clicked(b):
        show_main_menu(current_user)

    # Attach events
    transaction_type.observe(on_type_change)
    add_category_button.on_click(on_add_category_clicked)
    submit_button.on_click(on_submit_clicked)
    back_button.on_click(on_back_clicked)
    show_category_button.on_click(on_show_category_click)
    show_history_button.on_click(on_show_history_click)

    update_category_options()

    # Display all components
    display(widgets.VBox([
        transaction_type,
        category_dropdown,
        widgets.HBox([new_category_input, add_category_button]),
        amount_input,
        account_dropdown,
        submit_button,
        output,
        show_category_button,  # Show Category List button
        show_history_button  # Show Transaction History button
    ]))

    # Optionally, show delete category and delete transaction buttons only after needed
    # show_delete_category(user_id, transaction_type.value)
    # show_delete_transaction(user_id)

    display(back_button)

def show_category_list(user_id):
    clear_output()

    # Display heading for Category List with the largest font size
    display(widgets.HTML(value="<h2 style='font-size:20px; font-weight:bold;'>Category List</h2>"))

    # Fetch income and expense categories separately
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Fetch income categories
    cursor.execute("SELECT name FROM categories WHERE user_id = ? AND type = 'income'", (user_id,))
    income_categories = [row[0] for row in cursor.fetchall()]

    # Fetch expense categories
    cursor.execute("SELECT name FROM categories WHERE user_id = ? AND type = 'expense'", (user_id,))
    expense_categories = [row[0] for row in cursor.fetchall()]

    conn.close()

    # Display Income categories with a smaller font size and the colon
    display(widgets.HTML(value="<h3 style='font-size:16px; font-weight:bold;'>Income Categories:</h3>"))
    for category in income_categories:
        display(widgets.HTML(value=f"<p style='font-size:12px;'>{category}</p>"))

    # Display Expense categories with a smaller font size and the colon
    display(widgets.HTML(value="<h3 style='font-size:16px; font-weight:bold;'>Expense Categories:</h3>"))
    for category in expense_categories:
        display(widgets.HTML(value=f"<p style='font-size:12px;'>{category}</p>"))

    # Create dropdown for category selection
    category_dropdown = widgets.Dropdown(
        options=[None] + income_categories + expense_categories,
        description="Category:",
        disabled=False
    )

    # Single delete button
    delete_button = widgets.Button(description="Delete Category", button_style='danger')

    # Function to handle category deletion
    def on_delete_clicked(b):
        category_name = category_dropdown.value
        if category_name:
            delete_category(user_id, category_name)
        else:
            print("❌ Please select a category to delete.")

    delete_button.on_click(on_delete_clicked)

    # Show the category list, and delete button
    display(category_dropdown, delete_button)

    # Back button
    back_button = widgets.Button(description="Back to Add Transaction")
    back_button.on_click(lambda b: show_add_transaction(user_id))
    display(back_button)

def delete_category(user_id, category_name):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Delete the category
    cursor.execute('''DELETE FROM categories WHERE user_id = ? AND name = ?''',
                   (user_id, category_name))
    conn.commit()
    conn.close()

    # Refresh the category list after deletion
    show_category_list(user_id)

def add_transaction(user_id, category, type, amount, account_id):
    now = datetime.now()
    date_str = now.strftime("%Y-%m-%d")
    time_str = now.strftime("%H:%M")

    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Insert into transactions table
    cursor.execute('''
        INSERT INTO transactions (user_id, category, type, amount, date, time, account_id)
        VALUES (?, ?, ?, ?, ?, ?, ?)
    ''', (user_id, category, type, amount, date_str, time_str, account_id))

    # Update account balance
    if type == 'income':
        cursor.execute("UPDATE accounts SET balance = balance + ? WHERE id = ?", (amount, account_id))
    else:
        cursor.execute("UPDATE accounts SET balance = balance - ? WHERE id = ?", (amount, account_id))

    conn.commit()
    conn.close()

def show_delete_category(user_id, transaction_type):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM categories WHERE user_id = ? AND type = ?",
                   (user_id, transaction_type.lower()))
    categories = [row[0] for row in cursor.fetchall()]
    conn.close()

    if not categories:
        display(widgets.HTML(value="No categories available for deletion."))
        return

    dropdown = widgets.Dropdown(options=categories, description="Category:")
    delete_btn = widgets.Button(description="Delete", button_style="danger")
    output = widgets.Output()

    def on_delete(b):
        with output:
            clear_output()
            confirm = widgets.Button(description="Confirm")
            cancel = widgets.Button(description="Cancel")

            def do_confirm(b2):
                conn = sqlite3.connect(db_path)
                cursor = conn.cursor()
                cursor.execute("DELETE FROM categories WHERE user_id = ? AND type = ? AND name = ?",
                               (user_id, transaction_type.lower(), dropdown.value))
                conn.commit()
                conn.close()
                print(f"🗑️ Category '{dropdown.value}' deleted.")
                update_category_options()
                show_add_transaction(user_id)

            def do_cancel(b2):
                show_add_transaction(user_id)

            display(widgets.HTML(value=f"Are you sure you want to delete category: <b>{dropdown.value}</b>?"))
            display(widgets.HBox([confirm, cancel]))

            confirm.on_click(do_confirm)
            cancel.on_click(do_cancel)

    delete_btn.on_click(on_delete)
    display(widgets.VBox([dropdown, delete_btn, output]))

def show_transaction_history(user_id):
    clear_output()

    # Widgets for date selection and transaction fetching
    start_input = widgets.DatePicker(description="Start Date:")
    end_input = widgets.DatePicker(description="End Date:")
    fetch_button = widgets.Button(description="Fetch Transactions")
    dropdown = widgets.Dropdown(description="Transaction:")
    delete_button = widgets.Button(description="Delete", button_style='danger')
    output = widgets.Output()

    def fetch_transactions(b):
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()

        # Modify SQL query to join transactions with accounts to get account name
        cursor.execute('''SELECT t.id, t.category, t.amount, t.type, t.date, t.time, a.account_name
                          FROM transactions t
                          JOIN accounts a ON t.account_id = a.id
                          WHERE t.user_id = ? AND t.date BETWEEN ? AND ?
                          ORDER BY t.date DESC, t.time DESC''',
                       (user_id, start_input.value.strftime("%Y-%m-%d"), end_input.value.strftime("%Y-%m-%d")))
        transactions = cursor.fetchall()
        conn.close()

        if not transactions:
            dropdown.options = []
            with output:
                clear_output()
                print("No transactions in this range.")
            return

        # Update the dropdown options with transaction info including account name
        options = [(
            f"[{t[3].capitalize()}] {t[1]} - {t[2]} (Account: {t[6]}) ({t[4]} {t[5]})", t[0]
        ) for t in transactions]
        dropdown.options = options

    fetch_button.on_click(fetch_transactions)

    def on_delete_clicked(user_id, transaction_id):
        """Handle the transaction deletion."""
        if transaction_id:
            conn = sqlite3.connect(db_path)
            cursor = conn.cursor()

            # Fetch transaction data before deleting to update balance
            cursor.execute("SELECT amount, type, account_id FROM transactions WHERE id = ?", (transaction_id,))
            row = cursor.fetchone()
            if row:
                amount, type, account_id = row
                if type == 'income':
                    cursor.execute("UPDATE accounts SET balance = balance - ? WHERE id = ?", (amount, account_id))
                else:
                    cursor.execute("UPDATE accounts SET balance = balance + ? WHERE id = ?", (amount, account_id))

            cursor.execute("DELETE FROM transactions WHERE id = ?", (transaction_id,))
            conn.commit()
            conn.close()

            # Refresh the transaction history view after deletion
            show_transaction_history(user_id)

    delete_button.on_click(lambda b: on_delete_clicked(user_id, dropdown.value))  # Calling the function when delete button is clicked

    display(widgets.VBox([start_input, end_input, fetch_button, dropdown, delete_button, output]))

    # Back to Add Transaction button
    back_button = widgets.Button(description="Back to Add Transaction")
    back_button.on_click(lambda b: show_add_transaction(user_id))  # Navigate back to Add Transaction

    display(back_button)

# Show add subscription
def show_add_subscription(user_id):
    clear_output()

    subscription_name = widgets.Text(description="Subscription Name:")
    category_input = widgets.Dropdown(description="Category:", options=get_expense_categories(user_id))  # Expense categories
    charge_dates_input = widgets.Text(description="Charge Dates (comma separated):")  # 1, 15, 31, etc.
    amount_input = widgets.FloatText(description="Amount:")

    # Updated to show account name and ID tuple
    account_input = widgets.Dropdown(
        description="Select Account:",
        options=[(f"{acc[1]} | Balance: {acc[2]}", acc[0]) for acc in view_accounts(user_id)]  # (account_name, account_id) tuple
    )

    submit_button = widgets.Button(description="Add Subscription")
    show_subscription_button = widgets.Button(description="Subscription List")  # New button for subscription list

    # Handler for the Subscription List button
    def on_show_subscription_list(b):
        show_subscription_list(user_id)  # Display existing subscriptions when clicked

    show_subscription_button.on_click(on_show_subscription_list)

    # Check subscription limit before adding a new one
    def check_subscription_limit(user_id):
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        cursor.execute('''SELECT COUNT(*) FROM subscriptions WHERE user_id = ?''', (user_id,))
        count = cursor.fetchone()[0]
        conn.close()
        return count

    def on_submit(b):
        # Ensure user doesn't have more than 5 subscriptions
        if check_subscription_limit(user_id) >= 5:
            print("❌ You can only have up to 5 subscriptions.")
            return

        # Validate charge dates (to handle 31st on shorter months, February, etc.)
        valid_dates = validate_subscription_date(charge_dates_input.value)

        if not valid_dates:
            print("❌ Invalid charge dates. Subscription not created.")
            return  # Stop the subscription creation if there are no valid dates

        selected_account_id = account_input.value  # Directly use the account_id
        add_subscription(
            user_id,
            subscription_name.value,
            category_input.value,
            charge_dates_input.value,
            amount_input.value,
            selected_account_id
        )

    submit_button.on_click(on_submit)

    display(subscription_name, category_input, charge_dates_input, amount_input, account_input, submit_button, show_subscription_button)

    back_button = widgets.Button(description="Back to Main Menu")
    back_button.on_click(lambda b: show_main_menu(current_user))  # Use current_user here
    display(back_button)

def show_subscription_list(user_id):
    clear_output()

    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute('''SELECT id, subscription_name, category, amount, charge_dates
                      FROM subscriptions WHERE user_id = ?''', (user_id,))
    subscriptions = cursor.fetchall()
    conn.close()

    if not subscriptions:
        display(widgets.HTML(value="<b>No subscriptions found.</b>"))
        return

    # Display each subscription
    for subscription in subscriptions:
        subscription_id, subscription_name, category, amount, charge_dates = subscription
        display(widgets.HTML(value=f"<b>{subscription_name}</b> | Category: {category} | Amount: {amount} | Charge Dates: {charge_dates}"))

    # Add a button for deleting subscriptions
    delete_button = widgets.Button(description="Delete Subscription", button_style='danger')
    delete_button.on_click(lambda b: show_delete_subscription(user_id, subscriptions))  # Pass subscriptions to delete function
    display(delete_button)

    # Back button to navigate to Add Subscription
    back_button = widgets.Button(description="Back to Add Subscription")
    back_button.on_click(lambda b: show_add_subscription(user_id))  # Go back to Add Subscription page
    display(back_button)

def show_delete_subscription(user_id, subscriptions):
    clear_output()

    # Create a dropdown to select a subscription for deletion
    subscription_names = [subscription[1] for subscription in subscriptions]  # Extract subscription names
    delete_subscription_dropdown = widgets.Dropdown(
        options=subscription_names,
        description="Subscription:",
        disabled=False
    )

    # Button to confirm the deletion
    confirm_button = widgets.Button(description="Confirm Delete", button_style="danger")
    cancel_button = widgets.Button(description="Cancel", button_style="info")

    # Handle confirmation of deletion
    def on_confirm_delete(b):
        selected_subscription_name = delete_subscription_dropdown.value
        subscription_id = next(subscription[0] for subscription in subscriptions if subscription[1] == selected_subscription_name)
        delete_subscription(subscription_id, user_id)  # Call delete function
        show_subscription_list(user_id)  # Refresh subscription list after deletion

    # Handle cancellation
    def on_cancel_delete(b):
        show_subscription_list(user_id)  # Go back to subscription list if canceled

    confirm_button.on_click(on_confirm_delete)
    cancel_button.on_click(on_cancel_delete)

    # Display the dropdown and buttons
    display(delete_subscription_dropdown, confirm_button, cancel_button)

# Function to delete subscription
def delete_subscription(subscription_id, user_id):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute('DELETE FROM subscriptions WHERE id = ? AND user_id = ?', (subscription_id, user_id))
    conn.commit()
    conn.close()

    print(f"🗑️ Subscription with ID {subscription_id} deleted successfully.")

# Add subscription function
def add_subscription(user_id, subscription_name, category, charge_dates, amount, account_id):
    # Check for duplicate subscription names
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute('''SELECT 1 FROM subscriptions WHERE user_id = ? AND subscription_name = ?''', (user_id, subscription_name))
    if cursor.fetchone():
        print("❌ Subscription name already exists. Please choose a different name.")
        conn.close()
        return

    # Validate charge dates (ensure they are comma-separated integers)
    charge_dates_list = charge_dates.split(',')
    for date in charge_dates_list:
        if not date.strip().isdigit():
            print(f"❌ Invalid charge date: {date}. Charge dates must be numbers.")
            conn.close()
            return

    # Check if the amount is a valid number
    if amount <= 0:
        print("❌ Amount must be greater than 0.")
        conn.close()
        return

    # Insert the subscription details into the database
    cursor.execute('''
        INSERT INTO subscriptions (user_id, subscription_name, category, amount, charge_dates)
        VALUES (?, ?, ?, ?, ?)
    ''', (user_id, subscription_name, category, amount, charge_dates))
    conn.commit()
    conn.close()

    # Add subscription transaction (this will be charged immediately if necessary)
    process_subscription_transaction(user_id, subscription_name, category, charge_dates, amount, account_id)

    print("✅ Subscription added successfully.")

# Date Validation for Subscription Charge
def validate_subscription_date(charge_dates):
    # Parse charge dates
    charge_dates_list = [int(d.strip()) for d in charge_dates.split(',')]

    # Get current month and year
    today = datetime.today()
    month = today.month
    year = today.year

    # Get the number of days in the current month
    _, last_day = monthrange(year, month)  # This gets the last day of the current month

    # Filter valid dates
    valid_dates = [date for date in charge_dates_list if date <= last_day]
    invalid_dates = [date for date in charge_dates_list if date > last_day]

    if invalid_dates:
        print(f"❌ Invalid charge dates: {', '.join(map(str, invalid_dates))}. These dates do not exist in the current month.")

    return valid_dates

# Process Subscription Transaction (Auto Charge)
def process_subscription_transaction(user_id, subscription_name, category, charge_dates, amount, account_id):
    # Validate charge dates (to handle 31st on shorter months, February, etc.)
    valid_dates = validate_subscription_date(charge_dates)

    if not valid_dates:
        return  # Do not proceed if there are no valid dates

    # Get the current date and time
    now = datetime.now()
    date_str = now.strftime("%Y-%m-%d")
    time_str = now.strftime("%H:%M")

    # Insert transactions for valid charge dates
    for charge_date in valid_dates:
        if now.day == charge_date:
            # Insert the subscription transaction
            conn = sqlite3.connect(db_path)
            cursor = conn.cursor()

            cursor.execute('''
                INSERT INTO transactions (user_id, category, type, amount, date, time, account_id)
                VALUES (?, ?, ?, ?, ?, ?, ?)
            ''', (user_id, category, 'expense', amount, date_str, time_str, account_id))

            # Update account balance based on the transaction type (expense)
            cursor.execute("UPDATE accounts SET balance = balance - ? WHERE id = ?", (amount, account_id))
            conn.commit()
            conn.close()

    print("✅ Subscription transaction processed successfully.")

# Show wishlist
def show_wishlist(user_id):
    clear_output()

    # Add a note explaining the piggy bank concept
    display(widgets.HTML(value="<p style='font-size:16px; color:#B22222;'>Welcome to your Wishlist Piggy Bank! Add items you desire, fund them separately from your bank accounts, and once funded, you can purchase them!</p>"))

    wishlist_items = view_wishlist(user_id)

    # If there are 5 wishlist items, display a message and prevent adding more
    if len(wishlist_items) >= 5:
        display(widgets.HTML(value="<b>You have reached the maximum of 5 wishlist items! You cannot add more wishlist items.</b>"))

    # Display each wishlist item
    for item in wishlist_items:
        # Fetch the updated progress from the database
        progress = float(item[5]) if isinstance(item[5], (int, float)) else 0.0
        progress_bar = widgets.FloatProgress(value=progress, min=0, max=100, description="Progress:")

        # Display the wishlist item details
        display(widgets.HTML(value=f"<b>{item[2]}</b> | Price: {item[3]} | Notes: {item[4]} | Link: {item[5]}"))
        display(progress_bar)

        # Show notification for fully funded items
        if progress >= 100:
            display(widgets.HTML(value=f"🎉 Your wishlist item <b>{item[2]}</b> is fully funded! Ready for purchase!"))

    # Option to add funds to existing wishlist
    fund_button = widgets.Button(description="Fund Wishlist", button_style='info')
    fund_button.on_click(lambda b: show_fund_wishlist(user_id))
    display(fund_button)

    # Show the "Add Wishlist Item" button (if there are fewer than 5 items)
    if len(wishlist_items) < 5:
        add_button = widgets.Button(description="Add Wishlist Item", button_style="info")
        add_button.on_click(lambda b: show_add_wishlist(user_id))  # Show the add wishlist form
        display(add_button)

    # Show delete wishlist item button if there are items
    delete_button = widgets.Button(description="Delete Wishlist Item", button_style='danger')
    delete_button.on_click(lambda b: show_delete_wishlist_item(user_id))
    display(delete_button)

    # Back to main menu button
    back_button = widgets.Button(description="Back to Main Menu")
    back_button.on_click(lambda b: show_main_menu(current_user))
    display(back_button)

# --- Add Wishlist Item ---
def show_add_wishlist(user_id):
    clear_output()

    item_name_input = widgets.Text(description="Item Name:")
    price_input = widgets.FloatText(description="Price:")
    notes_input = widgets.Text(description="Notes:")
    link_input = widgets.Text(description="Link:")
    submit_button = widgets.Button(description="Add Wishlist")

    output = widgets.Output()

    def on_submit(b):
        # Add wishlist item to the database
        add_wishlist(user_id, item_name_input.value, price_input.value, notes_input.value, link_input.value)
        show_wishlist(user_id)  # Show updated wishlist after addition

    submit_button.on_click(on_submit)

    display(item_name_input, price_input, notes_input, link_input, submit_button, output)

    back_button = widgets.Button(description="Back to Main Menu")
    back_button.on_click(lambda b: show_main_menu(current_user))
    display(back_button)

# --- Add Wishlist Item to Database ---
def add_wishlist(user_id, item_name, price, notes, link):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute('''
    INSERT INTO wishlist (user_id, item_name, price, notes, link)
    VALUES (?, ?, ?, ?, ?)''', (user_id, item_name, price, notes, link))
    conn.commit()
    conn.close()

# --- View Wishlist Items ---
def view_wishlist(user_id):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute('''SELECT * FROM wishlist WHERE user_id = ?''', (user_id,))
    items = cursor.fetchall()
    conn.close()

    return items

# --- Delete Wishlist Item ---
def show_delete_wishlist_item(user_id):
    clear_output()

    # Fetch wishlist items
    wishlist_items = view_wishlist(user_id)

    if not wishlist_items:
        display(widgets.HTML(value="<b>No wishlist items to delete.</b>"))
        return

    # Create dropdown for wishlist items
    item_names = [item[2] for item in wishlist_items]  # Extract item names
    delete_dropdown = widgets.Dropdown(
        options=item_names,
        description="Wishlist Item:",
        disabled=False
    )

    delete_button = widgets.Button(description="Delete", button_style="danger")

    # Function to handle deletion
    def on_delete_clicked(b):
        selected_item_name = delete_dropdown.value
        item_id = next(item[0] for item in wishlist_items if item[2] == selected_item_name)
        delete_wishlist_item(item_id, user_id)

    delete_button.on_click(on_delete_clicked)

    display(delete_dropdown, delete_button)

    # Back to wishlist button
    back_button = widgets.Button(description="Back to Wishlist")
    back_button.on_click(lambda b: show_wishlist(user_id))  # Go back to wishlist
    display(back_button)

# --- Delete Wishlist Item from Database ---
def delete_wishlist_item(item_id, user_id):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute('DELETE FROM wishlist WHERE id = ? AND user_id = ?', (item_id, user_id))
    conn.commit()
    conn.close()

    # Refresh wishlist after deletion
    show_wishlist(user_id)

# --- Fund Existing Wishlist Item ---
def show_fund_wishlist(user_id):
    clear_output()

    wishlist_items = view_wishlist(user_id)

    if not wishlist_items:
        display(widgets.HTML(value="<b>No wishlist items found. Please create a wishlist item first.</b>"))
        return

    item_names = [item[2] for item in wishlist_items]  # Item names for dropdown

    # Dropdown to select wishlist item
    wishlist_dropdown = widgets.Dropdown(
        options=item_names,
        description="Select Wishlist:",
        disabled=False
    )

    amount_input = widgets.FloatText(description="Amount to Fund:")

    submit_button = widgets.Button(description="Fund Wishlist")

    def on_fund_submit(b):
        selected_item_name = wishlist_dropdown.value
        amount_to_fund = amount_input.value

        selected_item = next(item for item in wishlist_items if item[2] == selected_item_name)

        # Extract progress and price
        progress = float(selected_item[5]) if isinstance(selected_item[5], (int, float)) else 0.0
        price = float(selected_item[3]) if isinstance(selected_item[3], (int, float)) else 0.0

        # Ensure progress doesn't exceed 100% when funding
        if progress >= 100:
            display(widgets.HTML(value="❌ Cannot fund, the wishlist is already fully funded."))
            return

        # Update progress in the database
        new_progress = progress + (amount_to_fund / price) * 100

        # Ensure that the new progress does not exceed 100%
        if new_progress > 100:
            new_progress = 100

        # Update progress in the database
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        cursor.execute('''UPDATE wishlist SET progress = ? WHERE id = ? AND user_id = ?''', (new_progress, selected_item[0], user_id))
        conn.commit()
        conn.close()

        print(f"✅ Funded successfully. New Progress: {new_progress}%")

        # Refresh the wishlist display with updated progress
        show_wishlist(user_id)

    display(wishlist_dropdown, amount_input, submit_button)

    back_button = widgets.Button(description="Back to Wishlist")
    back_button.on_click(lambda b: show_wishlist(user_id))
    display(back_button)

def update_wishlist_funding(user_id, wishlist_id, amount_to_fund):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Fetch the current progress and price
    cursor.execute('''SELECT price, progress FROM wishlist WHERE id = ? AND user_id = ?''', (wishlist_id, user_id))
    result = cursor.fetchone()

    if result is None:
        print("❌ Wishlist item not found.")
        conn.close()
        return

    price, current_progress = result

    # Ensure price is greater than 0 to prevent division by zero
    if price <= 0:
        print("❌ Invalid price.")
        conn.close()
        return

    # Calculate new progress
    new_progress = current_progress + (amount_to_fund / price) * 100

    # Ensure progress doesn't exceed 100%
    if new_progress > 100:
        new_progress = 100

    # Update the progress in the database
    cursor.execute('''UPDATE wishlist SET progress = ? WHERE id = ? AND user_id = ?''', (new_progress, wishlist_id, user_id))
    conn.commit()
    conn.close()

    print(f"✅ Wishlist item progress updated to {new_progress}%")

# --- Run the Wishlist Feature ---
if current_user is not None:
    show_wishlist(current_user[0])  # Pass the user ID to display their wishlist
else:
    print("❌ No user is currently logged in.")

# Show dashboard
def show_dashboard(user_id):
    clear_output()

    # Dropdown to choose the month
    month_input = widgets.Dropdown(
        options=['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December'],
        description='Select Month:',
        disabled=False
    )

    submit_button = widgets.Button(description="Show Transactions")

    def on_submit(b):
        show_transactions_by_month(user_id, month_input.value)

    submit_button.on_click(on_submit)
    display(month_input, submit_button)

    back_button = widgets.Button(description="Back to Main Menu")
    back_button.on_click(lambda b: show_main_menu(current_user))  # Use current_user here
    display(back_button)

def show_transactions_by_month(user_id, month):
    # Convert month to the corresponding number (e.g., January -> 01)
    month_map = {
        'January': '01', 'February': '02', 'March': '03', 'April': '04',
        'May': '05', 'June': '06', 'July': '07', 'August': '08',
        'September': '09', 'October': '10', 'November': '11', 'December': '12'
    }
    selected_month = month_map[month]

    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Fetch the transactions for the selected month
    cursor.execute('''
        SELECT category, amount, type, date, time FROM transactions
        WHERE user_id = ? AND strftime('%m', date) = ?
    ''', (user_id, selected_month))
    transactions = cursor.fetchall()

    # Create a DataFrame for better table formatting
    if transactions:
        df = pd.DataFrame(transactions, columns=['Category', 'Amount', 'Type', 'Date', 'Time'])
        display(df)  # Using display() to show the DataFrame in a tabular format
    else:
        display(widgets.HTML(value="<b>No transactions found for this month.</b>"))

    # Create Pie and Line charts for the transactions
    generate_charts(transactions)

    conn.close()


def generate_charts(transactions):
    # Data Preparation
    categories = []
    amounts = []
    dates = []
    income_amounts = []
    expense_amounts = []
    income_dates = []
    expense_dates = []

    for transaction in transactions:
        categories.append(transaction[0])
        amounts.append(transaction[1])
        dates.append(transaction[3])

        if transaction[2] == "income":
            income_amounts.append(transaction[1])
            income_dates.append(transaction[3])
        else:
            expense_amounts.append(transaction[1])
            expense_dates.append(transaction[3])

    # Ensure that both x and y data have the same length
    if len(income_amounts) != len(income_dates):
        print("❌ Mismatch between income dates and income amounts.")
        return
    if len(expense_amounts) != len(expense_dates):
        print("❌ Mismatch between expense dates and expense amounts.")
        return

    # Pie chart for Income vs. Expense distribution
    fig, ax = plt.subplots(1, 1, figsize=(6, 6))
    labels = ['Income', 'Expense']
    sizes = [sum(income_amounts), sum(expense_amounts)]

    if sizes[0] == 0 and sizes[1] == 0:
        print("❌ No income or expense data to display.")
        return

    ax.pie(sizes, labels=labels, autopct='%1.1f%%', startangle=90)
    ax.axis('equal')  # Equal aspect ratio ensures the pie chart is circular.
    plt.title('Income vs Expense Distribution')
    plt.show()

    # Line chart for transaction trend (Income and Expense over time)
    fig, ax = plt.subplots(figsize=(10, 6))

    # Check for valid date times
    try:
        income_date_times = [datetime.strptime(date, "%Y-%m-%d") for date in income_dates]
        expense_date_times = [datetime.strptime(date, "%Y-%m-%d") for date in expense_dates]
    except ValueError as e:
        print(f"❌ Error parsing dates: {e}")
        return

    ax.plot_date(income_date_times, income_amounts, label='Income', linestyle='-', color='green')
    ax.plot_date(expense_date_times, expense_amounts, label='Expense', linestyle='-', color='red')

    # Formatting the date on x-axis
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    plt.xticks(rotation=45)

    ax.set_xlabel('Date')
    ax.set_ylabel('Amount')
    ax.set_title('Income and Expense Trend Over Time')
    ax.legend()

    plt.tight_layout()
    plt.show()

# Show badges
# Add the badge visuals dictionary
badge_visuals = {
    '5 Day Streak': {'color': 'green', 'icon': '🏅', 'shape': 'circle'},
    '10 Day Streak': {'color': 'blue', 'icon': '🏆', 'shape': 'star'},
    'First Account Created': {'color': 'yellow', 'icon': '💳', 'shape': 'rectangle'},
    '5 Accounts Created': {'color': 'purple', 'icon': '💼', 'shape': 'diamond'},
    'First Budget Created': {'color': 'red', 'icon': '📊', 'shape': 'circle'},
    '5 Budgets Created': {'color': 'orange', 'icon': '📈', 'shape': 'square'},
    '10th Transaction': {'color': 'pink', 'icon': '💸', 'shape': 'hexagon'},
    '100th Transaction': {'color': 'gold', 'icon': '💰', 'shape': 'circle'},
    'First Wishlist Item': {'color': 'aqua', 'icon': '📝', 'shape': 'star'},
    'First Dashboard Access': {'color': 'gray', 'icon': '📊', 'shape': 'rectangle'}
}

# Show badges with visuals
def show_badges(user_id):
    clear_output()

    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    cursor.execute('''SELECT badge_name, badge_description, date_awarded FROM badges WHERE user_id = ?''', (user_id,))
    badges = cursor.fetchall()

    if not badges:
        display(widgets.HTML(value="<b>No badges earned yet. Keep using the app to earn badges!</b>"))
    else:
        display(widgets.HTML(value="<h2>Your Badges:</h2>"))
        for badge in badges:
            badge_name = badge[0]
            badge_description = badge[1]
            badge_date = badge[2]

            # Get the visual information for this badge from the mapping
            visual = badge_visuals.get(badge_name, {'color': 'black', 'icon': '⭐', 'shape': 'circle'})

            # Customize the HTML to change the appearance based on visual
            badge_html = f"""
            <div style="display: inline-block; padding: 10px; margin: 10px; text-align: center;
                        background-color: {visual['color']}; border-radius: 10px;">
                <div style="font-size: 30px;">{visual['icon']}</div>
                <div><b>{badge_name}</b></div>
                <div>{badge_description}</div>
                <div><i>Awarded on {badge_date}</i></div>
            </div>
            """
            display(widgets.HTML(value=badge_html))

    conn.close()

    # Back button to return to the main menu
    back_button = widgets.Button(description="Back to Main Menu")
    back_button.on_click(lambda b: show_main_menu(current_user))
    display(back_button)

# Generalized function for awarding badges
def award_badge(user_id, badge_name, badge_description, threshold, entity="accounts"):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Check if the user already has this badge
    cursor.execute('''SELECT COUNT(*) FROM badges WHERE user_id = ? AND badge_name = ?''', (user_id, badge_name))
    existing_badge = cursor.fetchone()[0]

    if existing_badge > 0:
        print(f"User {user_id} already has the badge {badge_name}. No need to award it again.")
        conn.close()
        return  # Skip awarding if the badge is already awarded

    cursor.execute(f"SELECT COUNT(*) FROM {entity} WHERE user_id = ?", (user_id,))
    count = cursor.fetchone()[0]

    # Debugging: Print badge awarding info
    print(f"Awarding badge {badge_name} for user {user_id}, threshold: {threshold}, current count: {count}")

    if count == threshold:
        cursor.execute('''INSERT INTO badges (user_id, badge_name, badge_description, date_awarded)
                           VALUES (?, ?, ?, ?)''', (user_id, badge_name, badge_description, datetime.today().strftime("%Y-%m-%d")))
        conn.commit()
        print(f"Badge {badge_name} awarded to user {user_id}.")  # Debug confirmation
    else:
        print(f"User {user_id} does not meet the threshold for badge {badge_name}.")  # Debug confirmation

    conn.close()

# Badge-awarding functions for specific conditions
def award_streak_badge(user_id, streak_count):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    if streak_count == 5:
        award_badge(user_id, '5 Day Streak', 'Awarded for logging in for 5 consecutive days.', 5)
    elif streak_count == 10:
        award_badge(user_id, '10 Day Streak', 'Awarded for logging in for 10 consecutive days.', 10)

    conn.close()

def award_first_account_badge(user_id):
    print(f"Awarding 'First Account Created' badge to user {user_id}")
    award_badge(user_id, 'First Account Created', 'Awarded for creating your first account.', 1)

def award_five_accounts_badge(user_id):
    award_badge(user_id, '5 Accounts Created', 'Awarded for creating 5 accounts.', 5)

def award_first_budget_badge(user_id):
    award_badge(user_id, 'First Budget Created', 'Awarded for creating your first budget.', 1, 'budgets')

def award_five_budgets_badge(user_id):
    award_badge(user_id, '5 Budgets Created', 'Awarded for creating 5 budgets.', 5, 'budgets')

def award_transaction_badges(user_id, transaction_count):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    if transaction_count == 10:
        award_badge(user_id, '10th Transaction', 'Awarded for creating your 10th transaction.', 10, 'transactions')
    elif transaction_count == 100:
        award_badge(user_id, '100th Transaction', 'Awarded for creating your 100th transaction.', 100, 'transactions')

    conn.close()

def award_first_wishlist_badge(user_id):
    award_badge(user_id, 'First Wishlist Item', 'Awarded for creating your first wishlist item.', 1, 'wishlist')

def award_dashboard_access_badge(user_id):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    cursor.execute('''SELECT COUNT(*) FROM badges WHERE user_id = ? AND badge_name = 'First Dashboard Access' ''', (user_id,))
    existing_badge = cursor.fetchone()[0]

    if existing_badge == 0:
        cursor.execute('''INSERT INTO badges (user_id, badge_name, badge_description, date_awarded)
                           VALUES (?, ?, ?, ?)''', (user_id, 'First Dashboard Access', 'Awarded for accessing the dashboard for the first time.', datetime.today().strftime("%Y-%m-%d")))

    conn.commit()
    conn.close()

def check_and_award_badges(user_id):
    print(f"Checking and awarding badges for user: {user_id}")
    streak_count = update_streak(user_id)
    award_streak_badge(user_id, streak_count)

    # Award badges based on conditions
    award_first_account_badge(user_id)
    award_five_accounts_badge(user_id)
    award_first_budget_badge(user_id)
    award_five_budgets_badge(user_id)

    # Transaction-related badges
    transaction_count = get_transaction_count(user_id)
    award_transaction_badges(user_id, transaction_count)

    # Wishlist and dashboard-related badges
    award_first_wishlist_badge(user_id)
    award_dashboard_access_badge(user_id)

# Example: Call this after account creation to check if the user qualifies for a badge
def after_account_creation(user_id):
    check_and_award_badges(user_id)

def get_transaction_count(user_id):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Fetch the number of transactions for this user
    cursor.execute('''SELECT COUNT(*) FROM transactions WHERE user_id = ?''', (user_id,))
    transaction_count = cursor.fetchone()[0]

    conn.close()
    return transaction_count

# --- Historical App Runner ---
# Automatic startup disabled in this publication copy.
# Configure a disposable database before deliberately running the prototype.
# create_db()
# show_auth_choice_screen()

# **Database Management**

In [ ]:
#CHECK THE CREATED USER ACCOUNT
def get_all_usernames():
    conn = sqlite3.connect(db_path)  # Use the Google Drive path for persistent storage
    cursor = conn.cursor()

    cursor.execute("SELECT username FROM users")  # Fetch all usernames from the users table
    usernames = cursor.fetchall()  # Get all the usernames as a list of tuples

    conn.close()

    if usernames:
        print("Registered username:")
        for username in usernames:
            print(f"- {username[0]}")  # Print each username with a bullet point
    else:
        print("No users found.")

# Developer utility: disabled in this publication copy.
# Run manually only against a disposable test database.
# get_all_usernames()

In [ ]:
# DELETE USER ACCOUNT
import sqlite3

def delete_user_by_username(username):
    conn = sqlite3.connect(db_path)  # Use the correct path to the database
    cursor = conn.cursor()

    # Check if user exists
    cursor.execute("SELECT * FROM users WHERE username = ?", (username,))
    user = cursor.fetchone()

    if user:
        confirm = input(f"Are you sure you want to delete the user '{username}'? (yes/no): ")
        if confirm.lower() == "yes":
            cursor.execute("DELETE FROM users WHERE username = ?", (username,))
            conn.commit()
            print(f"User '{username}' deleted successfully.")
        else:
            print("Deletion canceled.")
    else:
        print(f"User '{username}' does not exist.")

    conn.close()

# Destructive developer utility: disabled in this publication copy.
# The historical function is retained for reference, not recommended use.
# delete_user_by_username("demo_user")